# Decision Trees 🌳

One of the most intuitive and interpretable machine learning algorithms. Think like a flowchart!

## What You'll Learn
- How decision trees work
- Building and visualizing trees
- Feature importance
- Overfitting problem
- Tree hyperparameters

## How Decision Trees Work 💭

A decision tree asks a series of yes/no questions to make decisions:

```
                    Is it sunny?
                    /          \
                  Yes          No
                  /              \
          Go outside         Is it warm?
                             /        \
                           Yes       No
                           /          \
                    Go to beach   Stay inside
```

- **Root node:** Starting question
- **Internal nodes:** Decision points
- **Leaves:** Final predictions
- **Splits:** Questions that divide the data

## Simple Decision Tree 🌳

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Simple dataset: Play tennis or not
weather_data = {
    'Temperature': [85, 80, 83, 70, 68, 65, 64, 72, 69, 75, 75, 72, 81, 81],
    'Humidity': [85, 90, 78, 96, 80, 70, 65, 95, 70, 80, 70, 90, 75, 80],
    'Wind': [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1],
    'Play_Tennis': [0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0]
}

df = pd.DataFrame(weather_data)
print("Dataset: Should we play tennis?")
print(df)

X = df[['Temperature', 'Humidity', 'Wind']]
y = df['Play_Tennis']

In [ ]:
# Build decision tree
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X, y)

print(f"Tree created with max_depth=3")
print(f"Accuracy: {tree.score(X, y):.1%}")

In [ ]:
# Visualize the tree
fig, ax = plt.subplots(figsize=(15, 10))
plot_tree(tree, 
          feature_names=['Temperature', 'Humidity', 'Wind'],
          class_names=['Don\'t Play', 'Play'],
          filled=True,
          ax=ax,
          fontsize=10)
plt.title('Decision Tree: Play Tennis?', fontsize=14)
plt.tight_layout()
plt.show()

print("Each box shows:")
print("- Split condition (e.g., Temperature <= 75.0)")
print("- Gini impurity (0=pure, 0.5=mixed)")
print("- Samples (how many data points)")
print("- Values (count of each class)")
print("- Color intensity (darker = more one class)")

## Feature Importance 🎯

In [ ]:
# Which features matter most?
importances = tree.feature_importances_
feature_names = ['Temperature', 'Humidity', 'Wind']

print("Feature Importance:")
for name, importance in zip(feature_names, importances):
    print(f"  {name:15s}: {importance:.2%}")

# Visualize
plt.figure(figsize=(8, 5))
plt.barh(feature_names, importances, color='skyblue', edgecolor='black')
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance', fontsize=14)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## The Overfitting Problem 🚨

In [ ]:
# Create more complex dataset
from sklearn.datasets import make_classification

X_complex, y_complex = make_classification(n_samples=200, n_features=2, n_informative=2, 
                                           n_redundant=0, random_state=42)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_complex, y_complex, 
                                                      test_size=0.3, random_state=42)

# Train trees with different depths
depths = [1, 3, 5, 10, 20, None]
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

print("Tree Performance vs Depth:")
print(f"\nDepth   | Train Acc | Test Acc  | Overfit Gap")
print("-" * 50)
for depth, train, test in zip(depths, train_scores, test_scores):
    gap = train - test
    print(f"{str(depth):6s} | {train:.1%}      | {test:.1%}     | {gap:.1%}")

In [ ]:
# Visualize overfitting
depth_labels = [str(d) if d else '∞' for d in depths]

plt.figure(figsize=(10, 6))
plt.plot(range(len(depths)), train_scores, marker='o', linewidth=2, label='Training Accuracy', color='green')
plt.plot(range(len(depths)), test_scores, marker='s', linewidth=2, label='Test Accuracy', color='red')
plt.xlabel('Tree Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Overfitting: Deep Trees Memorize, Shallow Trees Generalize', fontsize=14)
plt.xticks(range(len(depths)), depth_labels)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([0.4, 1.05])
plt.tight_layout()
plt.show()

print("\n📈 Observations:")
print("- Training accuracy: Increases with depth (tree memorizes)")
print("- Test accuracy: Peaks, then decreases (overfitting)")
print("- Gap grows: Deep trees don't generalize")

## Controlling Overfitting 🛡️

In [ ]:
# Use optimal depth
optimal_depth = 3  # From our analysis

tree_optimal = DecisionTreeClassifier(max_depth=optimal_depth, random_state=42)
tree_optimal.fit(X_train, y_train)

train_acc = tree_optimal.score(X_train, y_train)
test_acc = tree_optimal.score(X_test, y_test)

print(f"Optimal Tree (max_depth={optimal_depth}):")
print(f"  Training accuracy: {train_acc:.1%}")
print(f"  Test accuracy:     {test_acc:.1%}")
print(f"  Gap:               {train_acc - test_acc:.1%}")
print("\n✅ Good balance between learning and generalization!")

In [ ]:
# Other hyperparameters to control overfitting
tree_constrained = DecisionTreeClassifier(
    max_depth=5,                    # Limit tree depth
    min_samples_split=5,            # Min samples to split a node
    min_samples_leaf=2,             # Min samples at leaf
    random_state=42
)

tree_constrained.fit(X_train, y_train)

train_acc = tree_constrained.score(X_train, y_train)
test_acc = tree_constrained.score(X_test, y_test)

print("Constrained Tree:")
print(f"  max_depth=5")
print(f"  min_samples_split=5")
print(f"  min_samples_leaf=2")
print(f"\n  Training accuracy: {train_acc:.1%}")
print(f"  Test accuracy:     {test_acc:.1%}")
print(f"  Gap:               {train_acc - test_acc:.1%}")

## Decision Trees for Regression 📉

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Regression data: Price vs Square Feet
X_reg = np.array([1000, 1200, 1500, 1800, 2000, 2200, 2500, 2800]).reshape(-1, 1)
y_reg = np.array([150000, 180000, 210000, 240000, 280000, 310000, 360000, 400000])

# Build trees with different depths
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, depth in zip(axes, [1, 3, 10]):
    tree_reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree_reg.fit(X_reg, y_reg)
    
    # Predictions on grid
    x_range = np.linspace(800, 3000, 300).reshape(-1, 1)
    y_pred = tree_reg.predict(x_range)
    
    ax.scatter(X_reg, y_reg, s=100, color='blue', label='Actual', alpha=0.7)
    ax.plot(x_range, y_pred, 'r-', linewidth=2, label='Tree')
    ax.set_xlabel('Square Feet')
    ax.set_ylabel('Price')
    ax.set_title(f'Tree Depth = {depth}')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))

plt.suptitle('Decision Tree Regression: Effect of Tree Depth', fontsize=14)
plt.tight_layout()
plt.show()

print("Note the step-like pattern - trees create rectangular regions!")

## Key Hyperparameters 🔧

- **max_depth:** Maximum tree depth (smaller = simpler)
- **min_samples_split:** Min samples needed to split node (larger = simpler)
- **min_samples_leaf:** Min samples in leaf node (larger = simpler)
- **max_features:** Features to consider at each split

**Rule:** Make trees simpler to avoid overfitting!

## Advantages vs Disadvantages 📊

### ✅ Advantages
- Easy to understand (explainable)
- Works with both classification and regression
- Handles non-linear relationships
- Feature importance built-in
- Fast to predict

### ❌ Disadvantages
- Prone to overfitting
- Unstable (small data changes → big tree changes)
- Biased toward dominant classes
- Creates step-like boundaries

## Key Takeaways 🎯

✅ Decision trees split data using simple rules

✅ Very interpretable (show decisions clearly)

✅ Deep trees overfit; control with max_depth

✅ Feature importance helps understand decisions

✅ Ensemble methods (forests) often work better

## Practice Exercise 💪

1. Build decision tree on iris dataset
2. Visualize the tree
3. Find feature importance
4. Test different max_depth values
5. Build regression tree

Challenge: Find optimal depth using train-test split!

## Next Steps 🚀

→ **Random Forests:** Combine many trees (better than single tree!)

→ **Gradient Boosting:** Build trees sequentially

→ **Projects:** Real datasets with trees